# CPU: ghép Qwen + BGE, xem bất đồng, xuất ZIP

Không tải model. Đọc và xác minh điểm của hai worker trên cùng pool, ghép bằng RRF
để tránh cộng trực tiếp hai thang điểm khác nhau. Chỉ xuất ZIP khi đủ 1.200 query ở cả hai model.
Nếu mới chạy pilot, tạo báo cáo và request nhưng **không xuất submission thiếu query**.

Mẫu bất đồng gồm: một model xếp trong top 20 nhưng model kia không, hoặc lệch ít nhất
30 hạng; giới hạn tối đa 40 ứng viên/query. Đây là heuristic review, không phải nhãn đúng/sai.

Tùy chọn: copy request path sang worker GPU với `ROLE='arbiter'`. Khi worker đó xong,
đặt `ARBITRATION_REQUEST` tại đây rồi chạy lại cell merge. Model thứ ba chỉ đổi thứ tự
giữa các ứng viên được hỏi, giữ nguyên vị trí của các ứng viên ngoài tranh chấp.

Giữ nguyên docs, kiểm tra spans là nguyên văn của chunk trong submission.
Không có corpus nguồn hoặc evaluator BTC; model đồng thuận không được gọi là GT.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, sys, os, subprocess, hashlib, time, threading
ROOT = Path('/content/drive/MyDrive/ViBioMIR')
SUBMISSION_ZIP = ROOT / 'submission_17.zip'
QUERY_FILE = ROOT / 'query.parquet'
CAMPAIGN = 'ensemble_v1'  # Same across all workers. Use a new campaign if changing shard count.
RUN_PARENT = ROOT / 'multimodel_evidence' / CAMPAIGN
assert SUBMISSION_ZIP.is_file(), f'Upload submission_17.zip: {SUBMISSION_ZIP}'
assert QUERY_FILE.is_file(), f'Upload query/query.parquet as {QUERY_FILE}'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow==22.0.0'], check=True)


In [ ]:
BASE_SOURCE = '"""Standalone, resumable Qwen scoring of submission evidence and optional subspans.\n\nSource verification here means exact slices of the input submission, not the corpus.\nNo dependency on frozen BGE producer code. No official relevance evaluator included.\n"""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport sqlite3\nimport zipfile\n\nMODEL_ID = \'Qwen/Qwen3-Reranker-0.6B\'\nREVISION = \'6e9e69830b95c52b5fd889b7690dda3329508de3\'\nMODEL_DATE = \'2025-06-09T05:59:29Z\'\nEXPECTED_QUERY_SHA256 = \'ce13508b893bbd6b9169679d540630bc37eb5950cad08b4d5e0e5d7063670c08\'\nEXPECTED_ZIP_SHA256 = \'0963641cb422e4223d6fc4b0c22eb09938f8b03c8227053cc7a398bf33e25dbd\'\nPREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n          \'and the Instruct provided. Note that the answer can only be "yes" or "no".\'\n          \'<|im_end|>\\n<|im_start|>user\\n\')\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\nINSTRUCTION = \'Given a web search query, retrieve relevant passages that answer the query\'\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef signature(value):\n    return digest(canonical(value).encode())\n\n\ndef signed(value):\n    return {**value, \'signature\': signature(value)}\n\n\ndef checked(value):\n    require(isinstance(value, dict) and value.get(\'signature\') ==\n            signature({k: v for k, v in value.items() if k != \'signature\'}), \'Record checksum differs\')\n    return value\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(canonical(value), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef generation_policy(include_spans=False):\n    return dict(version=\'sentence-windows-v1\', include_spans=include_spans,\n                min_parent_chars=350, min_span_chars=100, sentence_windows=[1, 2, 3],\n                max_subspans_per_parent=4)\n\n\ndef model_policy():\n    return dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                instruction=INSTRUCTION, max_pair_tokens=4096, truncation=\'reject\',\n                scoring=\'logit_yes_minus_logit_no\', dtype=\'float16\',\n                attention=\'sdpa\', template_version=\'official-transformers-prefix-suffix-v1\')\n\n\ndef sentence_spans(text):\n    """Conservative boundaries; preserve offsets and avoid splitting decimals/abbreviations on \'.\'."""\n    boundaries = [0]\n    for match in re.finditer(r\'\\n+|[!?。！？]+[ \\t]*\', text):\n        boundaries.append(match.end())\n    # Split full stops only before a likely next sentence. Decimals like 0.10 stay intact.\n    for match in re.finditer(r\'\\.(?:[ \\t]+)(?=[A-ZÀ-Ỹ])\', text):\n        boundaries.append(match.end())\n    boundaries = sorted(set(boundaries + [len(text)]))\n    result = []\n    for a, b in zip(boundaries, boundaries[1:]):\n        while a < b and text[a].isspace():\n            a += 1\n        while b > a and text[b-1].isspace():\n            b -= 1\n        if a < b:\n            result.append((a, b))\n    return result\n\n\ndef evenly_choose(values, count):\n    if len(values) <= count:\n        return values\n    if count == 1:\n        return [values[len(values)//2]]\n    return [values[round(i * (len(values)-1)/(count-1))] for i in range(count)]\n\n\ndef validate_generation(policy):\n    require(policy[\'version\'] == \'sentence-windows-v1\' and type(policy[\'include_spans\']) is bool,\n            \'Unsupported generation policy\')\n    require(all(type(policy[k]) is int and policy[k] > 0 for k in\n                (\'min_parent_chars\', \'min_span_chars\', \'max_subspans_per_parent\')), \'Positive span budgets required\')\n    require(policy[\'sentence_windows\'] and all(type(n) is int and n > 0 for n in policy[\'sentence_windows\']),\n            \'Positive sentence-window sizes required\')\n\n\ndef make_candidates(row, policy):\n    validate_generation(policy)\n    candidates = []\n    for index, parent in enumerate(row[\'relevant_chunks\']):\n        text = parent[\'chunk_text\']\n        spans = [(0, len(text), \'original\')]\n        if policy[\'include_spans\'] and len(text) >= policy[\'min_parent_chars\']:\n            sentences = sentence_spans(text)\n            extras = set()\n            for width in policy[\'sentence_windows\']:\n                for start in range(len(sentences)-width+1):\n                    a, b = sentences[start][0], sentences[start+width-1][1]\n                    if b-a >= policy[\'min_span_chars\'] and b-a < len(text):\n                        extras.add((a, b))\n            spans.extend((a, b, \'subspan\') for a, b in evenly_choose(\n                sorted(extras), policy[\'max_subspans_per_parent\']))\n        parent_hash = digest(text.encode())\n        for a, b, kind in spans:\n            value = dict(parent_index=index, parent_text_sha256=parent_hash,\n                         doc_id=parent[\'doc_id\'], start=a, end=b, kind=kind,\n                         text_sha256=digest(text[a:b].encode()))\n            candidates.append({**value, \'candidate_id\': signature(value)})\n    return candidates\n\n\ndef evidence_text(row, candidate):\n    require(type(candidate[\'parent_index\']) is int and 0 <= candidate[\'parent_index\'] < len(row[\'relevant_chunks\']),\n            \'Unknown parent index\')\n    parent = row[\'relevant_chunks\'][candidate[\'parent_index\']]\n    text = parent[\'chunk_text\']\n    a, b = candidate[\'start\'], candidate[\'end\']\n    require(type(a) is int and type(b) is int and 0 <= a < b <= len(text), \'Invalid evidence offsets\')\n    require(parent[\'doc_id\'] == candidate[\'doc_id\'] and digest(text.encode()) == candidate[\'parent_text_sha256\']\n            and digest(text[a:b].encode()) == candidate[\'text_sha256\'], \'Evidence differs from parent submission\')\n    return text[a:b]\n\n\ndef load_inputs(submission_zip, query_file, expected_zip=EXPECTED_ZIP_SHA256,\n                expected_query=EXPECTED_QUERY_SHA256, expected_queries=1200):\n    require(file_hash(submission_zip) == expected_zip, \'Wrong input ZIP; select the original submission_17.zip\')\n    require(file_hash(query_file) == expected_query, \'Official query file differs\')\n    with zipfile.ZipFile(submission_zip) as z:\n        require(z.namelist() == [\'submission.json\'], \'Expected exactly one root submission.json\')\n        raw = z.read(\'submission.json\')\n    rows = json.loads(raw)\n    import pyarrow.parquet as pq\n    queries = pq.read_table(query_file, columns=[\'id\', \'query\']).to_pylist()\n    validate_input_rows(rows, queries, expected_queries)\n    return rows, queries, dict(submission_zip_sha256=expected_zip, submission_json_sha256=digest(raw),\n                               query_sha256=expected_query), raw\n\n\ndef validate_input_rows(rows, queries, expected_queries):\n    require(isinstance(rows, list) and len(rows) == len(queries) == expected_queries, \'Query count differs\')\n    ids = [q[\'id\'] for q in queries]\n    require(all(type(i) is int for i in ids) and len(set(ids)) == len(ids) and [r[\'id\'] for r in rows] == ids,\n            \'Query IDs/order differ or duplicate IDs\')\n    for row, query in zip(rows, queries):\n        require(isinstance(query[\'query\'], str) and query[\'query\'].strip(), \'Empty query\')\n        docs = row[\'relevant_docs\']\n        require(all(type(d) is int for d in docs) and len(set(docs)) == len(docs), \'Invalid doc list\')\n        for i, c in enumerate(row[\'relevant_chunks\']):\n            require(type(c[\'doc_id\']) is int and c[\'doc_id\'] in docs and isinstance(c[\'chunk_text\'], str)\n                    and c[\'chunk_text\'].strip() and type(c[\'chunk_order\']) is int and c[\'chunk_order\'] == i,\n                    \'Invalid parent evidence\')\n\n\ndef run_identity(inputs, generation, policy):\n    validate_generation(generation)\n    require(policy[\'model_id\'] == MODEL_ID and policy[\'revision\'] == REVISION and policy[\'commit_date\'] == MODEL_DATE,\n            \'Use the pinned Qwen checkpoint\')\n    return dict(operation=\'qwen-submission-rescore-v1\', inputs=inputs, generation=generation,\n                model_policy=policy, code_sha256=file_hash(__file__), source_scope=\'input_submission_only\')\n\n\ndef formatted_input(query, passage, policy):\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    return PREFIX + body + SUFFIX\n\n\ndef encoded_pair(tokenizer, query, passage, policy):\n    # Exactly the official prefix/body/suffix assembly, with no silent truncation.\n    body = f"<Instruct>: {policy[\'instruction\']}\\n<Query>: {query}\\n<Document>: {passage}"\n    ids = (tokenizer.encode(PREFIX, add_special_tokens=False)\n           + tokenizer.encode(body, add_special_tokens=False)\n           + tokenizer.encode(SUFFIX, add_special_tokens=False))\n    require(len(ids) <= policy[\'max_pair_tokens\'],\n            f"Pair has {len(ids)} tokens > {policy[\'max_pair_tokens\']}; increase policy budget in a new run")\n    return dict(input_ids=ids, attention_mask=[1]*len(ids))\n\n\nclass QwenScorer:\n    def __init__(self, policy, batch_size=8, padded_tokens=8192):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        require(torch.cuda.is_available(), \'Select a CUDA GPU runtime\')\n        require(policy[\'max_pair_tokens\'] <= 32768 and policy[\'max_pair_tokens\'] > 0\n                and policy[\'truncation\'] == \'reject\' and policy[\'scoring\'] == \'logit_yes_minus_logit_no\'\n                and policy[\'dtype\'] == \'float16\' and policy[\'attention\'] == \'sdpa\', \'Unsupported scoring policy\')\n        require(batch_size > 0 and padded_tokens >= policy[\'max_pair_tokens\'], \'Invalid GPU budgets\')\n        self.torch, self.policy = torch, policy\n        self.device = torch.device(\'cuda\')\n        self.batch_size, self.padded_tokens = batch_size, padded_tokens\n        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION,\n                                                       padding_side=\'left\', trust_remote_code=False)\n        self.model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,\n            torch_dtype=torch.float16, attn_implementation=\'sdpa\', trust_remote_code=False).to(\'cuda\').eval()\n        require(getattr(self.model.config, \'_commit_hash\', None) == REVISION, \'Loaded model revision differs\')\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.encode(\'yes\', add_special_tokens=False)\n        self.no = self.tokenizer.encode(\'no\', add_special_tokens=False)\n        require(len(self.yes) == len(self.no) == 1 and self.yes != self.no, \'Unexpected yes/no tokenizer mapping\')\n        count = sum(p.numel() for p in self.model.parameters())\n        require(0 < count <= 15_000_000_000 and policy[\'max_pair_tokens\'] <= self.model.config.max_position_embeddings,\n                \'Model size/context exceeds allowed budget\')\n        self.registry = dict(model_id=MODEL_ID, revision=REVISION, commit_date=MODEL_DATE,\n                             parameters=count, dtype=\'float16\', scoring=policy[\'scoring\'],\n                             transformers=transformers.__version__, torch=torch.__version__,\n                             tokenizer_revision=REVISION, gpu=torch.cuda.get_device_name(0))\n\n    def score(self, query, passages):\n        features = [encoded_pair(self.tokenizer, query, p, self.policy) for p in passages]\n        scores = []\n        index, budget = 0, self.batch_size\n        while index < len(features):\n            n = min(budget, len(features)-index)\n            while n > 1 and max(len(v[\'input_ids\']) for v in features[index:index+n])*n > self.padded_tokens:\n                n -= 1\n            try:\n                result = self._batch_scores(features[index:index+n])\n            except self.torch.cuda.OutOfMemoryError:\n                self.torch.cuda.empty_cache()\n                if n == 1:\n                    raise\n                budget = max(1, n//2)\n                print(f\'CUDA OOM: retry smaller batch={budget}\', flush=True)\n                continue\n            scores.extend(result)\n            index += n\n        return scores\n\n    def _batch_scores(self, features):\n        inputs = self.tokenizer.pad(features, padding=True, return_tensors=\'pt\')\n        inputs = {k: v.to(self.device) for k, v in inputs.items()}\n        with self.torch.inference_mode():\n            # Keep only the last token logits: avoids full B x sequence x vocabulary allocation.\n            logits = self.model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()\n            values = (logits[:, self.yes[0]] - logits[:, self.no[0]]).cpu().tolist()\n        require(all(math.isfinite(s) for s in values), \'Nonfinite Qwen score\')\n        return values\n\n\ndef cache_key(query, text, policy):\n    return signature(dict(query_sha256=digest(query.encode()), text_sha256=digest(text.encode()), policy=policy))\n\n\ndef verify_score_record(record, row, query, candidates, run_sig, policy):\n    checked(record)\n    require(record[\'id\'] == row[\'id\'] == query[\'id\'] and record[\'run_signature\'] == run_sig and\n            record[\'query_sha256\'] == digest(query[\'query\'].encode()) and\n            record[\'candidates_sha256\'] == signature(candidates), \'Score query/input identity differs\')\n    registry = record[\'model_registry\']\n    require(registry[\'model_id\'] == policy[\'model_id\'] and registry[\'revision\'] == policy[\'revision\'] and\n            registry[\'commit_date\'] == policy[\'commit_date\'] and\n            registry[\'tokenizer_revision\'] == policy[\'revision\'] and registry[\'scoring\'] == policy[\'scoring\'] and\n            registry[\'dtype\'] == policy[\'dtype\'] and type(registry[\'parameters\']) is int and\n            0 < registry[\'parameters\'] <= 15_000_000_000, \'Score model registry differs\')\n    require(len(record[\'scores\']) == len(candidates), \'Missing candidate scores\')\n    for c, score in zip(candidates, record[\'scores\']):\n        require(score[\'candidate_id\'] == c[\'candidate_id\'] and type(score[\'score\']) in (int, float)\n                and math.isfinite(score[\'score\']), \'Invalid score rows\')\n    return [v[\'score\'] for v in record[\'scores\']]\n\n\ndef run_scoring(rows, queries, identity, output, local_cache, *, stop_after_queries=5,\n                batch_size=8, padded_tokens=8192, scorer_factory=None):\n    require(stop_after_queries is None or type(stop_after_queries) is int and stop_after_queries > 0,\n            \'Positive pilot limit or None required\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path = output / \'manifest.json\'\n    run_sig = signature(identity)\n    if manifest_path.exists():\n        previous = checked(read(manifest_path))\n        require(previous[\'identity\'] == identity, \'Scoring inputs changed; use new run directory\')\n    else:\n        require(not any(output.iterdir()), \'Nonempty run without manifest; use new directory\')\n    done, hashes, new_count, scored_pairs = [], {}, 0, 0\n    scorer = None\n    def save(state):\n        write(manifest_path, signed(dict(identity=identity, run_signature=run_sig, state=state,\n            completed_query_ids=done, score_files=hashes, queries=len(rows),\n            new_queries_last_invocation=new_count, scored_unique_pairs_last_invocation=scored_pairs)))\n    save(\'running\')\n    Path(local_cache).parent.mkdir(parents=True, exist_ok=True)\n    with sqlite3.connect(local_cache) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,score REAL NOT NULL)\')\n        try:\n            for row, query in zip(rows, queries):\n                candidates = make_candidates(row, identity[\'generation\'])\n                path = output / f"query-{row[\'id\']}.json"\n                if path.is_file():\n                    verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n                else:\n                    keys, texts, missing = [], {}, []\n                    for c in candidates:\n                        text = evidence_text(row, c)\n                        key = cache_key(query[\'query\'], text, identity[\'model_policy\'])\n                        keys.append(key)\n                        texts[key] = text\n                    values = {}\n                    for key in dict.fromkeys(keys):\n                        hit = db.execute(\'SELECT score FROM scores WHERE key=?\', (key,)).fetchone()\n                        if hit:\n                            require(math.isfinite(hit[0]), \'Nonfinite cached score\')\n                            values[key] = hit[0]\n                        else:\n                            missing.append(key)\n                    if scorer is None:\n                        scorer = (scorer_factory or QwenScorer)(identity[\'model_policy\'],\n                            batch_size=batch_size, padded_tokens=padded_tokens)\n                    if missing:\n                        scores = scorer.score(query[\'query\'], [texts[k] for k in missing])\n                        require(len(scores) == len(missing) and all(type(s) in (int, float) and math.isfinite(s) for s in scores),\n                                \'Scorer returned invalid scores\')\n                        for key, score in zip(missing, scores):\n                            db.execute(\'INSERT INTO scores VALUES(?,?)\', (key, float(score)))\n                            values[key] = float(score)\n                        db.commit()\n                        scored_pairs += len(missing)\n                    record = signed(dict(id=row[\'id\'], run_signature=run_sig,\n                        query_sha256=digest(query[\'query\'].encode()), candidates_sha256=signature(candidates),\n                        model_registry=scorer.registry,\n                        scores=[dict(candidate_id=c[\'candidate_id\'], score=values[k]) for c,k in zip(candidates,keys)]))\n                    verify_score_record(record, row, query, candidates, run_sig, identity[\'model_policy\'])\n                    write(path, record)\n                    new_count += 1\n                done.append(row[\'id\']); hashes[str(row[\'id\'])] = file_hash(path)\n                save(\'running\')\n                print(f"Query {row[\'id\']}: {len(done)}/{len(rows)} complete; new this call={new_count}", flush=True)\n                if stop_after_queries is not None and new_count >= stop_after_queries and len(done) < len(rows):\n                    save(\'paused\')\n                    return read(manifest_path)\n        except BaseException:\n            save(\'interrupted\')\n            raise\n    save(\'complete\')\n    return read(manifest_path)\n\n\ndef overlap_iou(a, b):\n    if (a[\'doc_id\'], a[\'parent_index\']) != (b[\'doc_id\'], b[\'parent_index\']):\n        return 0.0  # Source coordinates across different parent chunks are unavailable.\n    overlap = max(0, min(a[\'end\'], b[\'end\']) - max(a[\'start\'], b[\'start\']))\n    return overlap / (max(a[\'end\'], b[\'end\']) - min(a[\'start\'], b[\'start\']))\n\n\ndef select_evidence(row, candidates, scores, limits, originals_only=False):\n    require(type(limits[\'max_chunks\']) is int and limits[\'max_chunks\'] > 0 and\n            type(limits[\'per_doc\']) is int and limits[\'per_doc\'] > 0 and\n            0 < limits[\'overlap_iou\'] <= 1, \'Invalid selection limits\')\n    require(len(candidates) == len(scores) and all(math.isfinite(s) for s in scores), \'Invalid score array\')\n    selected, per_doc, decisions, texts = [], {}, [], {}\n    ranked = sorted(zip(candidates, scores), key=lambda x: (-x[1], x[0][\'candidate_id\']))\n    for c, score in ranked:\n        text = evidence_text(row, c)\n        reason = None\n        if originals_only and c[\'kind\'] != \'original\':\n            reason = \'subspan_disabled\'\n        elif len(selected) >= limits[\'max_chunks\']:\n            reason = \'global_cap\'\n        elif per_doc.get(c[\'doc_id\'], 0) >= limits[\'per_doc\']:\n            reason = \'per_doc_cap\'\n        else:\n            peers = [v for v in selected if v[\'doc_id\'] == c[\'doc_id\']]\n            if any(text == texts[v[\'candidate_id\']] for v in peers):\n                reason = \'same_doc_exact_duplicate\'\n            elif any(overlap_iou(c, v) >= limits[\'overlap_iou\'] for v in peers):\n                reason = \'same_parent_overlap\'\n        if reason is None:\n            selected.append(c)\n            texts[c[\'candidate_id\']] = text\n            per_doc[c[\'doc_id\']] = per_doc.get(c[\'doc_id\'], 0)+1\n        decisions.append(dict(candidate_id=c[\'candidate_id\'], reason=reason or \'selected\', score=score))\n    prediction = dict(id=row[\'id\'], relevant_docs=list(row[\'relevant_docs\']), relevant_chunks=[\n        dict(doc_id=c[\'doc_id\'], chunk_text=texts[c[\'candidate_id\']], chunk_order=i) for i,c in enumerate(selected)])\n    return prediction, selected, decisions\n\n\ndef package_variants(rows, queries, identity, score_dir, destination, baseline_raw, variants):\n    score_dir, destination = Path(score_dir), Path(destination)\n    require(not destination.exists(), \'Choose a new export directory\')\n    require(digest(baseline_raw) == identity[\'inputs\'][\'submission_json_sha256\'], \'Baseline bytes differ\')\n    manifest = checked(read(score_dir / \'manifest.json\'))\n    require(manifest[\'identity\'] == identity and manifest[\'state\'] == \'complete\' and\n            manifest[\'completed_query_ids\'] == [r[\'id\'] for r in rows], \'Finish all scoring queries before export\')\n    run_sig = signature(identity)\n    manifest_hash = file_hash(score_dir / \'manifest.json\')\n    destination.mkdir(parents=True)\n    # S0 is byte-for-byte baseline JSON, repackaged without changing prediction data.\n    with zipfile.ZipFile(destination / \'S0_baseline.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'submission.json\', baseline_raw)\n    predictions, counts = {v[\'name\']: [] for v in variants}, {v[\'name\']: [] for v in variants}\n    require(len(predictions) == len(variants) and all(re.fullmatch(r\'[A-Za-z0-9_-]+\', v[\'name\']) for v in variants),\n            \'Unique safe variant names required\')\n    evidence_sidecar = destination / \'EVIDENCE_PROVENANCE.jsonl\'\n    with evidence_sidecar.open(\'w\', encoding=\'utf-8\') as sidecar:\n        for row, query in zip(rows, queries):\n            candidates = make_candidates(row, identity[\'generation\'])\n            path = score_dir / f"query-{row[\'id\']}.json"\n            require(file_hash(path) == manifest[\'score_files\'][str(row[\'id\'])], \'Score file changed after completion\')\n            scores = verify_score_record(read(path), row, query, candidates, run_sig, identity[\'model_policy\'])\n            for variant in variants:\n                pred, selected, decisions = select_evidence(row, candidates, scores, variant[\'limits\'],\n                                                           variant[\'originals_only\'])\n                require(pred[\'relevant_docs\'] == row[\'relevant_docs\'], \'Document list changed\')\n                predictions[variant[\'name\']].append(pred)\n                before = {(c[\'doc_id\'], c[\'chunk_text\']) for c in row[\'relevant_chunks\']}\n                after = {(c[\'doc_id\'], c[\'chunk_text\']) for c in pred[\'relevant_chunks\']}\n                counts[variant[\'name\']].append(dict(id=row[\'id\'], chunks=len(selected),\n                    changed_evidence=before != after, subspans=sum(c[\'kind\']==\'subspan\' for c in selected),\n                    documents_with_chunks=len({c[\'doc_id\'] for c in selected})))\n                sidecar.write(canonical(dict(variant=variant[\'name\'], id=row[\'id\'], selected=selected,\n                                             decisions=decisions))+\'\\n\')\n    require(file_hash(score_dir / \'manifest.json\') == manifest_hash, \'Scoring manifest changed during export\')\n    exports = []\n    for variant in variants:\n        name = variant[\'name\']\n        payload = canonical(predictions[name]).encode()\n        archive = destination / f\'{name}.zip\'\n        with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'submission.json\', payload)\n        with zipfile.ZipFile(archive) as z:\n            require(z.namelist() == [\'submission.json\'] and z.read(\'submission.json\') == payload, \'ZIP differs\')\n        exports.append(dict(variant=variant, zip=archive.name, zip_sha256=file_hash(archive),\n            json_sha256=digest(payload), queries=len(rows),\n            changed_queries=sum(c[\'changed_evidence\'] for c in counts[name]),\n            total_chunks=sum(c[\'chunks\'] for c in counts[name]),\n            total_subspans=sum(c[\'subspans\'] for c in counts[name]), query_statistics=counts[name]))\n    report = dict(validation=\'passed\', validation_scope=\'input submission subset/slices and score integrity\',\n        original_corpus_evidence_verified=False, official_doc_allowlist_verified=False,\n        relevance_measured=False, official_lcs_evaluated=False, source_identity=identity,\n        score_manifest_sha256=manifest_hash, baseline_json_sha256=digest(baseline_raw),\n        provenance_sha256=file_hash(evidence_sidecar), exports=exports)\n    write(destination / \'REPORT.json\', report)\n    return report\n'
ENSEMBLE_SOURCE = '"""Parallel evidence rerankers, rank fusion and optional disagreement arbitration.\n\nAll labels are model judgments. Exact source scope is the input submission only.\n"""\nimport math\nfrom pathlib import Path\nimport sqlite3\nimport zipfile\n\ntry:\n    from . import qwen_submission_trial as core\nexcept ImportError:\n    import qwen_submission_trial as core\n\n\ndef policies():\n    qwen = core.model_policy()\n    return dict(qwen=qwen,\n        bge=dict(model_id=\'BAAI/bge-reranker-v2-m3\',\n            revision=\'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e\', commit_date=\'2024-06-24T14:08:45Z\',\n            max_pair_tokens=4096, truncation=\'reject\', scoring=\'sequence_classification_logit\',\n            dtype=\'float16\', attention=\'sdpa\', template_version=\'tokenizer-pair-v1\'),\n        arbiter={**qwen, \'model_id\': \'Qwen/Qwen3-Reranker-4B\',\n            \'revision\': \'22e683669bc0f0bd69640a1354a6d0aebcfeede5\', \'commit_date\': \'2026-04-16T08:56:07Z\'})\n\n\ndef build_plan(inputs, generation):\n    core.validate_generation(generation)\n    return core.signed(dict(schema=\'multimodel-evidence-v1\', inputs=inputs, generation=generation,\n        models=policies(), code_sha256=core.file_hash(__file__), base_code_sha256=core.file_hash(core.__file__),\n        scope=\'input-submission-evidence-only\', ground_truth=False))\n\n\ndef validate_plan(plan, inputs):\n    core.checked(plan)\n    core.require(plan == build_plan(inputs, plan[\'generation\']), \'Input, model or support code differs from plan\')\n\n\ndef worker_identity(plan, role, shard_index, shard_count, request=None):\n    core.require(role in plan[\'models\'] and type(shard_index) is int and type(shard_count) is int\n        and 1 <= shard_count <= 16 and 0 <= shard_index < shard_count, \'Invalid role/shard assignment\')\n    core.require((role == \'arbiter\') == (request is not None), \'Arbiter needs a verified disagreement request\')\n    return dict(operation=\'multimodel-evidence-worker\', plan_signature=plan[\'signature\'], role=role,\n                shard_index=shard_index, shard_count=shard_count,\n                request_signature=request[\'signature\'] if request else None)\n\n\ndef worker_directory(root, identity):\n    scope = identity[\'request_signature\'] or \'all\'\n    return Path(root) / \'workers\' / identity[\'role\'] / scope / (\n        f"part-{identity[\'shard_index\']}-of-{identity[\'shard_count\']}")\n\n\ndef bge_pair(tokenizer, query, passage, policy):\n    features = dict(tokenizer(query, passage, add_special_tokens=True, truncation=False,\n                              return_token_type_ids=False))\n    core.require(len(features[\'input_ids\']) <= policy[\'max_pair_tokens\'],\n                 f"BGE pair exceeds {policy[\'max_pair_tokens\']} tokens; use a new larger-budget plan")\n    return features\n\n\nclass ModelScorer(core.QwenScorer):\n    """Same score() batching for two architectures; independent model inputs/registries."""\n    def __init__(self, policy, batch_size=8, padded_tokens=8192):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer\n        core.require(policy in policies().values(), \'Use a pinned ensemble model policy\')\n        core.require(torch.cuda.is_available(), \'Select CUDA GPU runtime\')\n        core.require(batch_size > 0 and padded_tokens >= policy[\'max_pair_tokens\'], \'Invalid GPU budgets\')\n        self.torch, self.device, self.policy = torch, torch.device(\'cuda\'), policy\n        self.batch_size, self.padded_tokens = batch_size, padded_tokens\n        self.is_bge = policy[\'model_id\'] == policies()[\'bge\'][\'model_id\']\n        self.tokenizer = AutoTokenizer.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            trust_remote_code=False, padding_side=\'right\' if self.is_bge else \'left\')\n        loader = AutoModelForSequenceClassification if self.is_bge else AutoModelForCausalLM\n        self.model = loader.from_pretrained(policy[\'model_id\'], revision=policy[\'revision\'],\n            torch_dtype=torch.float16, attn_implementation=\'sdpa\', trust_remote_code=False,\n            use_safetensors=True).to(self.device).eval()\n        core.require(getattr(self.model.config, \'_commit_hash\', None) == policy[\'revision\'], \'Loaded revision differs\')\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.yes = self.tokenizer.encode(\'yes\', add_special_tokens=False)\n        self.no = self.tokenizer.encode(\'no\', add_special_tokens=False)\n        if self.is_bge:\n            core.require(self.model.config.num_labels == 1, \'BGE requires one classification logit\')\n        else:\n            core.require(len(self.yes) == len(self.no) == 1 and self.yes != self.no, \'Unexpected yes/no token IDs\')\n        count = sum(p.numel() for p in self.model.parameters())\n        context = self.model.config.max_position_embeddings - (2 if self.is_bge else 0)\n        core.require(0 < count <= 15_000_000_000 and policy[\'max_pair_tokens\'] <= context, \'Model/context limit exceeded\')\n        self.registry = dict(model_id=policy[\'model_id\'], revision=policy[\'revision\'],\n            tokenizer_revision=policy[\'revision\'], commit_date=policy[\'commit_date\'], parameters=count,\n            dtype=policy[\'dtype\'], scoring=policy[\'scoring\'], torch=torch.__version__,\n            transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))\n\n    def score(self, query, passages):\n        if not self.is_bge:\n            return super().score(query, passages)\n        features = [bge_pair(self.tokenizer, query, p, self.policy) for p in passages]\n        scores, index, budget = [], 0, self.batch_size\n        while index < len(features):\n            n = min(budget, len(features)-index)\n            while n > 1 and max(len(v[\'input_ids\']) for v in features[index:index+n])*n > self.padded_tokens:\n                n -= 1\n            try:\n                values = self._batch_scores(features[index:index+n])\n            except self.torch.cuda.OutOfMemoryError:\n                self.torch.cuda.empty_cache()\n                if n == 1:\n                    raise\n                budget = max(1, n//2)\n                print(\'CUDA OOM: retry BGE batch\', budget, flush=True)\n                continue\n            scores.extend(values); index += n\n        return scores\n\n    def _batch_scores(self, features):\n        if not self.is_bge:\n            return super()._batch_scores(features)\n        inputs = {k: v.to(self.device) for k,v in self.tokenizer.pad(features,padding=True,return_tensors=\'pt\').items()}\n        with self.torch.inference_mode():\n            values = self.model(**inputs, return_dict=True).logits.reshape(-1).float().cpu().tolist()\n        core.require(len(values) == len(features) and all(math.isfinite(x) for x in values), \'Invalid BGE logits\')\n        return values\n\n\ndef verify_record(record, row, query, candidates, identity, policy):\n    # Compatible score envelope, but worker identity binds the model/plan/shard separately.\n    return core.verify_score_record(record, row, query, candidates, core.signature(identity), policy)\n\n\ndef tasks_for(rows, queries, plan, identity, request=None):\n    lookup = {t[\'id\']: t for t in request[\'tasks\']} if request else None\n    tasks = []\n    for index, (row, query) in enumerate(zip(rows, queries)):\n        if index % identity[\'shard_count\'] != identity[\'shard_index\']:\n            continue\n        if lookup is not None and row[\'id\'] not in lookup:\n            continue\n        candidates = core.make_candidates(row, plan[\'generation\'])\n        if lookup is not None:\n            t = lookup[row[\'id\']]\n            core.require(t[\'candidates_sha256\'] == core.signature(candidates), \'Arbiter candidate set differs\')\n            allowed = set(t[\'requested_candidate_ids\'])\n            candidates = [c for c in candidates if c[\'candidate_id\'] in allowed]\n            core.require(len(candidates) == len(allowed), \'Unknown arbitration candidate\')\n        if candidates:\n            tasks.append((row, query, candidates))\n    return tasks\n\n\ndef validate_request(request, plan, root):\n    core.checked(request)\n    core.require(request[\'plan_signature\'] == plan[\'signature\'] and request[\'schema\'] == \'disagreement-request-v1\',\n                 \'Request belongs to another plan\')\n    core.require(len({t[\'id\'] for t in request[\'tasks\']}) == len(request[\'tasks\']), \'Duplicate request query\')\n    for t in request[\'tasks\']:\n        core.require(len(set(t[\'requested_candidate_ids\'])) == len(t[\'requested_candidate_ids\']), \'Duplicate request candidate\')\n        for source in t[\'source_records\']:\n            path = (Path(root) / source[\'path\']).resolve()\n            core.require(Path(root).resolve() in path.parents and core.file_hash(path) == source[\'sha256\'],\n                         \'Input model scores changed since arbitration request\')\n\n\ndef import_old_qwen(directory, row, query, candidates, plan):\n    """Reuse only proven identical query/passage/policy scores from the prior notebook."""\n    if directory is None:\n        return None\n    directory = Path(directory)\n    manifest = core.checked(core.read(directory / \'manifest.json\'))\n    old = manifest[\'identity\']\n    core.require(old[\'inputs\'] == plan[\'inputs\'] and old[\'generation\'] == plan[\'generation\'] and\n        old[\'model_policy\'] == plan[\'models\'][\'qwen\'] and manifest[\'run_signature\'] == core.signature(old),\n        \'Existing Qwen run differs in input, spans or scoring policy; disable reuse or select matching run\')\n    path = directory / f"query-{row[\'id\']}.json"\n    if not path.is_file() or row[\'id\'] not in manifest[\'completed_query_ids\']:\n        return None\n    core.require(core.file_hash(path) == manifest[\'score_files\'][str(row[\'id\'])], \'Old Qwen score file changed\')\n    record = core.read(path)\n    core.verify_score_record(record,row,query,candidates,manifest[\'run_signature\'],plan[\'models\'][\'qwen\'])\n    return record, dict(path=str(path), sha256=core.file_hash(path), old_run_signature=manifest[\'run_signature\'])\n\n\ndef score_worker(rows, queries, plan, root, role, *, shard_index=0, shard_count=1,\n                 request=None, stop_after_queries=20, local_cache=None, batch_size=8,\n                 padded_tokens=8192, reuse_qwen=None, scorer_factory=None):\n    core.require(stop_after_queries is None or type(stop_after_queries) is int and stop_after_queries>0, \'Invalid pilot limit\')\n    validate_plan(plan, plan[\'inputs\'])\n    if request:\n        validate_request(request,plan,root)\n    identity = worker_identity(plan,role,shard_index,shard_count,request)\n    output = worker_directory(root,identity)\n    output.mkdir(parents=True,exist_ok=True)\n    manifest_path = output/\'manifest.json\'\n    if manifest_path.exists():\n        previous = core.checked(core.read(manifest_path))\n        core.require(previous[\'identity\'] == identity,\'Worker identity differs\')\n    else:\n        core.require(not any(output.iterdir()),\'Nonempty worker directory has no manifest\')\n    tasks = tasks_for(rows,queries,plan,identity,request)\n    policy = plan[\'models\'][role]\n    done, hashes, new, scorer = [], {}, 0, None\n    def save(state):\n        core.write(manifest_path,core.signed(dict(identity=identity,state=state,\n            expected_query_ids=[r[\'id\'] for r,_,_ in tasks],completed_query_ids=done,\n            score_files=hashes,new_queries_last_invocation=new)))\n    save(\'running\')\n    local_cache = Path(local_cache or Path(root)/f\'cache-{role}.sqlite3\')\n    local_cache.parent.mkdir(parents=True,exist_ok=True)\n    with sqlite3.connect(local_cache) as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,score REAL NOT NULL)\')\n        try:\n            for row,query,candidates in tasks:\n                path=output/f"query-{row[\'id\']}.json"\n                if path.is_file():\n                    verify_record(core.read(path),row,query,candidates,identity,policy)\n                else:\n                    imported = import_old_qwen(reuse_qwen,row,query,candidates,plan) if role==\'qwen\' else None\n                    if imported:\n                        old, provenance = imported\n                        values, registry, provenance = [s[\'score\'] for s in old[\'scores\']],old[\'model_registry\'],provenance\n                    else:\n                        texts = [core.evidence_text(row,c) for c in candidates]\n                        keys=[core.cache_key(query[\'query\'],text,policy) for text in texts]\n                        unique = dict(zip(keys,texts)); known={}; missing=[]\n                        for key in unique:\n                            hit=db.execute(\'SELECT score FROM scores WHERE key=?\',(key,)).fetchone()\n                            if hit:\n                                core.require(math.isfinite(hit[0]),\'Invalid cached value\')\n                                known[key]=hit[0]\n                            else:\n                                missing.append(key)\n                        if scorer is None:\n                            scorer=(scorer_factory or ModelScorer)(policy,batch_size=batch_size,padded_tokens=padded_tokens)\n                        if missing:\n                            values=scorer.score(query[\'query\'],[unique[k] for k in missing])\n                            core.require(len(values)==len(missing) and all(type(s) in (int,float) and math.isfinite(s) for s in values),\n                                         \'Invalid model output\')\n                            for key,value in zip(missing,values):\n                                db.execute(\'INSERT INTO scores VALUES(?,?)\',(key,float(value))); known[key]=float(value)\n                            db.commit()\n                        values=[known[k] for k in keys]; registry=scorer.registry; provenance=None\n                    record=core.signed(dict(id=row[\'id\'],run_signature=core.signature(identity),\n                        query_sha256=core.digest(query[\'query\'].encode()),candidates_sha256=core.signature(candidates),\n                        model_registry=registry,scores=[dict(candidate_id=c[\'candidate_id\'],score=s) for c,s in zip(candidates,values)],\n                        imported_from=provenance))\n                    verify_record(record,row,query,candidates,identity,policy)\n                    core.write(path,record); new+=1\n                done.append(row[\'id\']); hashes[str(row[\'id\'])]=core.file_hash(path); save(\'running\')\n                print(f"{role} part {shard_index}: query {row[\'id\']} complete {len(done)}/{len(tasks)}, new={new}",flush=True)\n                if stop_after_queries is not None and new>=stop_after_queries and len(done)<len(tasks):\n                    save(\'paused\'); return core.read(manifest_path)\n        except BaseException:\n            save(\'interrupted\'); raise\n    save(\'complete\')\n    return core.read(manifest_path)\n\n\ndef collect_role(rows, queries, plan, root, role, request=None):\n    scope = request[\'signature\'] if request else \'all\'\n    manifests = sorted((Path(root)/\'workers\'/role/scope).glob(\'*/manifest.json\'))\n    records, shard_counts = {},set()\n    by_id={r[\'id\']:(r,q) for r,q in zip(rows,queries)}\n    if request:\n        validate_request(request,plan,root)\n    for path in manifests:\n        m=core.checked(core.read(path)); ident=m[\'identity\']\n        core.require(ident[\'role\']==role and ident==worker_identity(plan,role,ident[\'shard_index\'],ident[\'shard_count\'],request),\n                     \'Worker plan/model/shard differs\')\n        core.require(path.parent.resolve()==worker_directory(root,ident).resolve(),\'Unexpected shard directory\')\n        shard_counts.add(ident[\'shard_count\'])\n        expected={r[\'id\']:cs for r,_,cs in tasks_for(rows,queries,plan,ident,request)}\n        core.require(m[\'expected_query_ids\']==list(expected) and len(set(m[\'completed_query_ids\']))==len(m[\'completed_query_ids\']),\n                     \'Worker query coverage differs\')\n        for qid in m[\'completed_query_ids\']:\n            core.require(qid in expected and qid not in records,\'Unknown or duplicated query across shards\')\n            row,q=by_id[qid]; score_path=path.parent/f\'query-{qid}.json\'\n            core.require(core.file_hash(score_path)==m[\'score_files\'][str(qid)],\'Score file changed\')\n            values=verify_record(core.read(score_path),row,q,expected[qid],ident,plan[\'models\'][role])\n            records[qid]=dict(candidates=expected[qid],scores=values,\n                source=dict(path=score_path.relative_to(root).as_posix(),sha256=core.file_hash(score_path)))\n    core.require(len(shard_counts)<=1,\'Mixed shard counts: use one consistent shard_count per model\')\n    return records\n\n\ndef ranks(candidates,scores):\n    core.require(len(candidates)==len(scores) and len({c[\'candidate_id\'] for c in candidates})==len(candidates)\n                 and all(math.isfinite(s) for s in scores),\'Invalid fusion candidates/scores\')\n    order=sorted(range(len(candidates)),key=lambda i:(-scores[i],candidates[i][\'candidate_id\']))\n    return {candidates[i][\'candidate_id\']:rank for rank,i in enumerate(order,1)}\n\n\ndef fused_scores(candidates, left, right, k=60):\n    core.require(type(k) is int and k>0,\'Positive RRF k required\')\n    a,b=ranks(candidates,left),ranks(candidates,right)\n    return [1/(k+a[c[\'candidate_id\']])+1/(k+b[c[\'candidate_id\']]) for c in candidates]\n\n\ndef disagreement_ids(candidates,left,right, *, top_k=20,rank_gap=30,limit=40):\n    a,b=ranks(candidates,left),ranks(candidates,right)\n    bad=[c[\'candidate_id\'] for c in candidates if\n         (a[c[\'candidate_id\']]<=top_k)!=(b[c[\'candidate_id\']]<=top_k)\n         or abs(a[c[\'candidate_id\']]-b[c[\'candidate_id\']])>=rank_gap]\n    bad.sort(key=lambda key:(-int((a[key]<=top_k)!=(b[key]<=top_k)),-abs(a[key]-b[key]),min(a[key],b[key]),key))\n    return bad[:limit]\n\n\ndef reorder_disputed_slots(candidates,base_scores,requested_ids,arbiter_scores):\n    """Arbiter compares ONLY requested candidates; never grants missing-score bonuses."""\n    by_id={c[\'candidate_id\']:c for c in candidates}\n    core.require(set(requested_ids)<=set(by_id) and len(set(requested_ids))==len(requested_ids)\n                 and len(requested_ids)==len(arbiter_scores) and all(math.isfinite(s) for s in arbiter_scores),\n                 \'Invalid arbitration score coverage\')\n    order=sorted(by_id,key=lambda key:(-base_scores[key],key))\n    slots=[i for i,key in enumerate(order) if key in set(requested_ids)]\n    judged=sorted(zip(requested_ids,arbiter_scores),key=lambda x:(-x[1],x[0]))\n    for index,(key,_) in zip(slots,judged):\n        order[index]=key\n    positions={key:i for i,key in enumerate(order)}\n    return [float(len(order)-positions[c[\'candidate_id\']]) for c in candidates]\n\n\ndef merge_and_export(rows,queries,plan,root,destination,baseline_raw, *, request=None,\n                     top_k=20,rank_gap=30,request_limit=40):\n    validate_plan(plan,plan[\'inputs\'])\n    core.require(core.digest(baseline_raw)==plan[\'inputs\'][\'submission_json_sha256\'],\'Baseline differs\')\n    core.require(type(top_k) is int and top_k>0 and type(rank_gap) is int and rank_gap>0 and\n                 type(request_limit) is int and request_limit>0,\'Invalid disagreement budgets\')\n    qwen=collect_role(rows,queries,plan,root,\'qwen\'); bge=collect_role(rows,queries,plan,root,\'bge\')\n    common={r[\'id\'] for r in rows}&set(qwen)&set(bge)\n    core.require(common,\'No shared completed queries: run both qwen and bge workers first\')\n    full=len(common)==len(rows)\n    arbitration=collect_role(rows,queries,plan,root,\'arbiter\',request) if request else {}\n    request_tasks={t[\'id\']:t for t in request[\'tasks\']} if request else {}\n    arbiter_full=request is not None and full and set(request_tasks)<=set(arbitration) and set(request[\'query_ids\'])==common\n    destination=Path(destination)\n    core.require(not destination.exists(),\'Choose a new merge destination\')\n    destination.mkdir(parents=True)\n    variants=[dict(name=\'Qwen_80_cap2\',kind=\'qwen\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8)),\n              dict(name=\'BGE_80_cap2\',kind=\'bge\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8)),\n              dict(name=\'RRF_120_cap6\',kind=\'rrf\',limits=dict(max_chunks=120,per_doc=6,overlap_iou=.8)),\n              dict(name=\'RRF_80_cap2\',kind=\'rrf\',limits=dict(max_chunks=80,per_doc=2,overlap_iou=.8))]\n    if arbiter_full:\n        variants += [dict(name=\'Arbitrated_120_cap6\',kind=\'arbiter\',limits=dict(max_chunks=120,per_doc=6,overlap_iou=.8))]\n    predictions={v[\'name\']:[] for v in variants}; stats={v[\'name\']:[] for v in variants}\n    tasks=[]; review_count=0\n    with (destination/\'DISAGREEMENT_REVIEW.jsonl\').open(\'w\',encoding=\'utf-8\') as review, \\\n         (destination/\'EVIDENCE_PROVENANCE.jsonl\').open(\'w\',encoding=\'utf-8\') as provenance:\n        for row,query in zip(rows,queries):\n            qid=row[\'id\']\n            if qid not in common:\n                continue\n            candidates=qwen[qid][\'candidates\']\n            core.require(candidates==bge[qid][\'candidates\'],\'Models scored different candidate pools\')\n            a,b=qwen[qid][\'scores\'],bge[qid][\'scores\']\n            rr=fused_scores(candidates,a,b)\n            requested=disagreement_ids(candidates,a,b,top_k=top_k,rank_gap=rank_gap,limit=request_limit)\n            if requested:\n                tasks.append(dict(id=qid,candidates_sha256=core.signature(candidates),requested_candidate_ids=requested,\n                                  source_records=[qwen[qid][\'source\'],bge[qid][\'source\']]))\n                ra,rb=ranks(candidates,a),ranks(candidates,b)\n                by_id={c[\'candidate_id\']:c for c in candidates}\n                review.write(core.canonical(dict(id=qid,query=query[\'query\'],model_judgments_not_gold=True,\n                    disputed_count=len(requested),examples=[dict(candidate=by_id[key],\n                    text=core.evidence_text(row,by_id[key]),qwen_rank=ra[key],bge_rank=rb[key]) for key in requested[:5]]))+\'\\n\')\n                review_count+=1\n            third=None\n            if arbiter_full and qid in arbitration:\n                task=request_tasks[qid]\n                core.require(task[\'candidates_sha256\']==core.signature(candidates),\'Arbiter full-pool hash differs\')\n                subset=arbitration[qid]\n                ids=[c[\'candidate_id\'] for c in subset[\'candidates\']]\n                third=reorder_disputed_slots(candidates,dict(zip([c[\'candidate_id\'] for c in candidates],rr)),ids,subset[\'scores\'])\n            for v in variants:\n                values={\'qwen\':a,\'bge\':b,\'rrf\':rr,\'arbiter\':third if third is not None else rr}[v[\'kind\']]\n                pred,selected,decisions=core.select_evidence(row,candidates,values,v[\'limits\'])\n                core.require(pred[\'relevant_docs\']==row[\'relevant_docs\'],\'Document list changed\')\n                predictions[v[\'name\']].append(pred)\n                before={(c[\'doc_id\'],c[\'chunk_text\']) for c in row[\'relevant_chunks\']}\n                after={(c[\'doc_id\'],c[\'chunk_text\']) for c in pred[\'relevant_chunks\']}\n                stats[v[\'name\']].append(dict(id=qid,chunks=len(selected),changed=before!=after,\n                    subspans=sum(c[\'kind\']==\'subspan\' for c in selected)))\n                provenance.write(core.canonical(dict(variant=v[\'name\'],id=qid,selected=selected,decisions=decisions))+\'\\n\')\n    new_request=core.signed(dict(schema=\'disagreement-request-v1\',plan_signature=plan[\'signature\'],\n        query_ids=[r[\'id\'] for r in rows if r[\'id\'] in common],scope=\'full\' if full else \'pilot\',\n        rules=dict(top_k=top_k,rank_gap=rank_gap,max_candidates_per_query=request_limit),tasks=tasks))\n    requests_dir=Path(root)/\'requests\'\n    requests_dir.mkdir(parents=True,exist_ok=True)\n    request_path=requests_dir/f"request-{new_request[\'signature\']}.json"\n    if request_path.exists():\n        core.require(core.read(request_path)==new_request,\'Existing request changed\')\n    else:\n        core.write(request_path,new_request)\n    exports=[]\n    if full:\n        with zipfile.ZipFile(destination/\'S0_baseline.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n            z.writestr(\'submission.json\',baseline_raw)\n        for v in variants:\n            payload=core.canonical(predictions[v[\'name\']]).encode()\n            path=destination/(v[\'name\']+\'.zip\')\n            with zipfile.ZipFile(path,\'w\',zipfile.ZIP_DEFLATED) as z:\n                z.writestr(\'submission.json\',payload)\n            with zipfile.ZipFile(path) as z:\n                core.require(z.namelist()==[\'submission.json\'] and z.read(\'submission.json\')==payload,\'Output ZIP differs\')\n            exports.append(dict(variant=v,zip=path.name,zip_sha256=core.file_hash(path),\n                json_sha256=core.digest(payload),queries=len(rows),\n                changed_queries=sum(s[\'changed\'] for s in stats[v[\'name\']]),\n                chunks=sum(s[\'chunks\'] for s in stats[v[\'name\']]),subspans=sum(s[\'subspans\'] for s in stats[v[\'name\']])))\n    report=dict(state=\'full_export\' if full else \'pilot_diagnostics_only\',plan_signature=plan[\'signature\'],\n        total_queries=len(rows),qwen_complete_queries=len(qwen),bge_complete_queries=len(bge),\n        shared_queries=len(common),queries_with_disagreements=review_count,\n        requested_pairs=sum(len(t[\'requested_candidate_ids\']) for t in tasks),request_path=str(request_path),\n        request_signature=new_request[\'signature\'],arbiter_complete=arbiter_full,\n        applied_arbitration_request_signature=request[\'signature\'] if arbiter_full else None,\n        original_corpus_verified=False,relevance_measured=False,official_lcs_evaluated=False,\n        model_judgments_are_ground_truth=False,source_scope=\'exact slices of input submission only\',\n        source_inputs=plan[\'inputs\'],baseline_json_sha256=core.digest(baseline_raw),exports=exports,\n        pilot_statistics=None if full else stats)\n    core.write(destination/\'REPORT.json\',report)\n    return report\n'
CODE_DIR = Path('/content/multimodel_evidence_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in [('qwen_submission_trial', BASE_SOURCE), ('multimodel_evidence', ENSEMBLE_SOURCE)]:
    if name in sys.modules:
        assert Path(sys.modules[name].__file__).read_text(encoding='utf-8') == source, 'Restart runtime to load new support code.'
    else:
        (CODE_DIR / (name + '.py')).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_DIR))
import qwen_submission_trial as core
from multimodel_evidence import build_plan, validate_plan, score_worker, merge_and_export
from contextlib import contextmanager
@contextmanager
def progress(label):
    stop = threading.Event()
    start = time.monotonic()
    def heartbeat():
        while not stop.wait(30):
            print(label, 'minutes:', round((time.monotonic()-start)/60, 1), flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    print(label, flush=True)
    thread.start()
    try:
        yield
    finally:
        stop.set()
        thread.join()


In [ ]:
with progress('Verify input ZIP, official query snapshot and evidence schema'):
    rows, queries, inputs, baseline_raw = core.load_inputs(SUBMISSION_ZIP, QUERY_FILE)
print('Queries:', len(rows), 'Input chunks:', sum(len(r['relevant_chunks']) for r in rows))


In [ ]:
PLAN_DIRECTORY = None  # Paste Path printed by GPU workers if several plans exist.
if PLAN_DIRECTORY is None:
    plans = []
    for path in sorted(RUN_PARENT.glob('*/plan.json')):
        value = core.read(path)
        print('Plan:', path.parent, 'spans:', value['generation']['include_spans'])
        if value['inputs'] == inputs:
            plans.append(path.parent)
    assert len(plans) == 1, 'Set PLAN_DIRECTORY explicitly to the shared worker plan'
    PLAN_DIRECTORY = plans[0]
RUN = Path(PLAN_DIRECTORY)
plan = core.read(RUN / 'plan.json')
validate_plan(plan, inputs)
ARBITRATION_REQUEST = None  # Optional Path to the request scored by the arbiter worker.
TOP_K_DISAGREEMENT = 20
RANK_GAP = 30
MAX_ARBITER_CANDIDATES_PER_QUERY = 40


In [ ]:
from datetime import datetime, timezone
DESTINATION = RUN / ('merge_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
request = core.read(ARBITRATION_REQUEST) if ARBITRATION_REQUEST is not None else None
with progress('Verify worker scores, merge ranks and select evidence'):
    report = merge_and_export(rows, queries, plan, RUN, DESTINATION, baseline_raw,
        request=request, top_k=TOP_K_DISAGREEMENT, rank_gap=RANK_GAP,
        request_limit=MAX_ARBITER_CANDIDATES_PER_QUERY)
print('State:', report['state'])
print('Qwen / BGE / shared:', report['qwen_complete_queries'], report['bge_complete_queries'], report['shared_queries'])
print('Disputed queries:', report['queries_with_disagreements'], 'Arbiter pairs:', report['requested_pairs'])
print('Request path:', report['request_path'])
print('Report and review:', DESTINATION)
for item in report['exports']:
    print(item['zip'], 'changed queries:', item['changed_queries'], 'chunks:', item['chunks'], 'subspans:', item['subspans'])
if not report['exports']:
    print('PILOT ONLY: finish all base workers, then rerun this merge cell; no submission ZIP was created.')
print('Relevance, official LCS and clinical correctness: not measured.')


In [ ]:
assert report['state'] == 'full_export', 'Pilot diagnostics cannot be submitted; finish both model workers first'
DOWNLOAD_NAME = 'RRF_80_cap2.zip'  # A trial configuration, not an automatically selected best result.
assert DOWNLOAD_NAME in ['S0_baseline.zip'] + [e['zip'] for e in report['exports']]
from google.colab import files
files.download(str(DESTINATION / DOWNLOAD_NAME))
